# 🔮 Vision Transformer & Mixture of Experts (MoE) Medical Segmentation Trainer

This notebook fine-tunes a **SegFormer** architecture (standard or dynamically replaced with a **Mixture of Experts (MoE) block**) on a custom medical image segmentation dataset (e.g. ISIC skin lesions, BUSI breast cancer scans, or any custom image-mask pairs).

Developed as part of advanced computer vision graduate research.

In [ ]:
# 1. Mount Google Drive to save checkpoints and plots persistently
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the Research Repository and Install Dependencies
%cd /content
!rm -rf vision_tranformer_moe
!git clone https://github.com/toqeer-ahmed/vision_tranformer_moe.git
%cd /content/vision_tranformer_moe
!pip install -r requirements.txt

In [ ]:
# 3. Download & Prepare Medical Dataset (Kvasir-SEG Polyp Segmentation)
# This script downloads the official Kvasir-SEG dataset (46MB), extracts it, and automatically structures it
# into the images and masks folder layout required by the framework.
import os
import urllib.request
import zipfile
import shutil
import ssl

# Bypass SSL Certificate Verification to avoid certificate verification errors
ssl._create_default_https_context = ssl._create_unverified_context

zip_url = "https://datasets.simula.no/downloads/kvasir-seg.zip"
zip_path = "kvasir-seg.zip"
dest_dir = "data/medical_dataset"

if os.path.exists(os.path.join(dest_dir, "images")) and os.path.exists(os.path.join(dest_dir, "masks")):
    print("Dataset already exists and is structured at data/medical_dataset/")
else:
    os.makedirs(dest_dir, exist_ok=True)
    print(f"Downloading Kvasir-SEG dataset from {zip_url}...")
    try:
        urllib.request.urlretrieve(zip_url, zip_path)
        print("Download complete. Extracting zip archive...")
        
        # Extract zip
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall("kvasir_temp")
        
        # Dynamically locate images/ and masks/ inside extracted temp folder
        extracted_root = None
        for root, dirs, files in os.walk("kvasir_temp"):
            if "images" in dirs and "masks" in dirs:
                extracted_root = root
                break
        
        if extracted_root:
            print(f"Found images and masks directories at: {extracted_root}")
            # Clean existing directories if any
            shutil.rmtree(os.path.join(dest_dir, "images"), ignore_errors=True)
            shutil.rmtree(os.path.join(dest_dir, "masks"), ignore_errors=True)
            
            # Move images and masks to final dest_dir
            shutil.move(os.path.join(extracted_root, "images"), os.path.join(dest_dir, "images"))
            shutil.move(os.path.join(extracted_root, "masks"), os.path.join(dest_dir, "masks"))
            print(f"Dataset successfully prepared and structured at {dest_dir}!")
        else:
            print("Error: Could not find 'images' and 'masks' directories inside the zip file.")
            
    except Exception as e:
        print(f"An error occurred during dataset preparation: {e}")
        
    finally:
        # Clean up temp files
        shutil.rmtree("kvasir_temp", ignore_errors=True)
        if os.path.exists(zip_path):
            os.remove(zip_path)

In [ ]:
# 4. Launch MoE-Segformer Training
# The model will dynamically load nvidia/mit-b0, swap standard MLPs for MoELayers, and train on the GPU.
!PYTHONPATH=. python training/train_moe.py --config configs/medical_segmentation.yaml

In [ ]:
# 5. Monitor Metrics in Real-time via Tensorboard
%load_ext tensorboard
%tensorboard --logdir outputs/medical_segmentation/logs

In [ ]:
# 6. Visualize Predictions and Training Curves
import matplotlib.pyplot as plt
from PIL import Image
import glob
import os

plots = glob.glob("outputs/medical_segmentation/plots/*.png")
if plots:
    print(f"Found {len(plots)} training curves and validation outputs:")
    for p in sorted(plots):
        print(f"\nDisplaying: {os.path.basename(p)}")
        img = Image.open(p)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("No plots found. Ensure training completes at least 1 epoch and validation IoU improves!")